# Multi-fidelity FLD regression: FIRE (TabPFN v3)

Same split as `mf_ar1_nargp_cms.ipynb`. LF TabPFN -> distribution features `[x, mu, var, q_0.1..q_0.9]` -> residual TabPFN on the HF rows. Writes `outputs/<kernel_set>/<repr>_<form>_<strategy>/<tag>/` (metrics JSON, prediction CSVs).

In [ ]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

ROOT = Path.cwd()   # notebook must be started from the cms folder
DATA = ROOT / "data"
assert DATA.is_dir(), f"no data folder under {ROOT}"

# L8 = 13 GSH coefficients, L12 = 37
REPR = "L8"
# training budgets count rows (texture x rho), LF larger than HF
N_LF_TRAIN, N_HF_TRAIN = 90, 45
N_HF_SUB = 30
FAR_REFERENCE = "hf"      # far subset: farthest from the HF training textures ("hf") or from LF U HF ("union")
STRATEGY = "random"       # or "local"
FORMULATION = "complete"  # or "sparse"
KERNEL_SET = "engineered"   # "engineered" / "standard"
REPEAT = 0
SEED = 0
TEST_SEED = 9000 + REPEAT
TRAIN_SEED = {"random": 100, "local": 200}[STRATEGY] * 1000 + REPEAT
TAG = (f"{REPR}_lf{N_LF_TRAIN}_hf{N_HF_TRAIN}_{STRATEGY}{REPEAT}_{FORMULATION}"
       + ("" if KERNEL_SET == "engineered" else "_" + KERNEL_SET))
OUTPUTS_NAME = "outputs"
OUT_DIR = ROOT / OUTPUTS_NAME / KERNEL_SET / f"{REPR}_{FORMULATION}_{STRATEGY}" / TAG
print("config:", TAG)

RHO_ARR = np.array([-0.5, -0.3, -0.1, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
N_RHO = len(RHO_ARR)
TARGETS = ("minor", "major")

in_lo = pd.read_csv(DATA / f"in_lo_{REPR}.csv").sort_values("index").reset_index(drop=True)
in_hi = pd.read_csv(DATA / f"in_hi_{REPR}.csv").sort_values("index").reset_index(drop=True)
out_lo = pd.read_csv(DATA / "out_lo_fld.csv").sort_values("index").reset_index(drop=True)
out_hi = pd.read_csv(DATA / "out_hi_fld.csv").sort_values("index").reset_index(drop=True)

feat_cols = [c for c in in_lo.columns if c not in ("filename", "index")]
# target columns come interleaved: minor_1, major_1, minor_2, ...
targ_cols = [c for c in out_lo.columns if c not in ("filename", "index")]

in_lo = in_lo[in_lo["index"].isin(out_lo["index"])].reset_index(drop=True)
in_hi = in_hi[in_hi["index"].isin(out_hi["index"])].reset_index(drop=True)
assert (in_lo["index"].values == out_lo["index"].values).all()
assert (in_hi["index"].values == out_hi["index"].values).all()

X_lf_all, Y_lf_all = in_lo[feat_cols].to_numpy(float), out_lo[targ_cols].to_numpy(float)
X_hf_all, Y_hf_all = in_hi[feat_cols].to_numpy(float), out_hi[targ_cols].to_numpy(float)
lf_index_vals = out_lo["index"].to_numpy()
hf_index_vals = out_hi["index"].to_numpy()

rng_train = np.random.default_rng(TRAIN_SEED)
rng_hf_sub = np.random.default_rng(TEST_SEED + 500)

def nearest_dist(A, B):
    d = np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(-1))
    return d.min(1)

# z-scored GSH coefficients (LF pool statistics), used by the sampling only
mu_all = X_lf_all.mean(0)
sd_all = X_lf_all.std(0)
sd_all[sd_all < 1e-12] = 1.0
Z_lf = (X_lf_all - mu_all) / sd_all
Z_hf = (X_hf_all - mu_all) / sd_all

# cluster seed for the local strategy; drawn in every case so the rng stream stays the same
seed_row = int(rng_train.choice(len(X_lf_all)))
Z_seed = Z_lf[seed_row]

def pick_train(cand, Zc, n_rows, max_tex=None, strategy=None):
    # returns the chosen texture ids (sorted) and the rows as (position in that list, rho index)
    if strategy is None:
        strategy = STRATEGY
    if FORMULATION == "complete":
        n_tex = n_rows // N_RHO
        if strategy == "random":
            tex = np.sort(rng_train.choice(cand, n_tex, replace=False))
        else:
            d = np.sqrt(((Zc - Z_seed) ** 2).sum(-1))
            tex = np.sort(cand[np.argsort(d)[:n_tex]])
        rows = np.array([(p, k) for p in range(n_tex) for k in range(N_RHO)], int)
        return tex, rows
    if strategy == "random":
        if max_tex is not None and len(cand) > max_tex:
            cand = np.sort(rng_train.choice(cand, max_tex, replace=False))
        tex = np.sort(cand)
        assert n_rows <= len(tex) * N_RHO, "sparse/random budget exceeds the available (texture, rho) cells"
        grid = rng_train.choice(len(tex) * N_RHO, n_rows, replace=False)
        rows = np.column_stack([grid // N_RHO, grid % N_RHO])
    else:
        # nearest textures first; rho cycled over the distance rank
        d = np.sqrt(((Zc - Z_seed) ** 2).sum(-1))
        tex = cand[np.argsort(d)]
        if max_tex is not None:
            tex = tex[:max_tex]
        T = len(tex)
        assert n_rows <= T * N_RHO, "sparse/local budget exceeds the available (texture, rho) cells"
        rows = np.array([(r % T, (r % T + r // T) % N_RHO) for r in range(n_rows)], int)
    used = np.unique(rows[:, 0])
    remap = np.full(rows[:, 0].max() + 1, -1)
    remap[used] = np.arange(len(used))
    rows[:, 0] = remap[rows[:, 0]]
    order = np.argsort(tex[used])
    inv = np.empty(len(used), int)
    inv[order] = np.arange(len(used))
    rows[:, 0] = inv[rows[:, 0]]
    return np.sort(tex[used]), rows

def build_train(X_all, Y_all, tex, rows):
    XL = np.column_stack([X_all[tex][rows[:, 0]], RHO_ARR[rows[:, 1]]])
    y = {"minor": Y_all[tex][rows[:, 0], 2 * rows[:, 1]],
         "major": Y_all[tex][rows[:, 0], 2 * rows[:, 1] + 1]}
    return XL, y

# LF training textures are drawn at random under both strategies; "local" applies to the HF textures
lf_tex, lf_rows = pick_train(np.arange(len(X_lf_all)), Z_lf, N_LF_TRAIN, strategy="random")
# sparse HF training must leave 2*N_HF_SUB textures untouched for the test subsets
HF_CAP = len(X_hf_all) - 2 * N_HF_SUB
hf_tex, hf_rows = pick_train(np.arange(len(X_hf_all)), Z_hf, N_HF_TRAIN, max_tex=HF_CAP)
XL_lf_tr, yL_lf_tr = build_train(X_lf_all, Y_lf_all, lf_tex, lf_rows)
XL_hf_tr, yL_hf_tr = build_train(X_hf_all, Y_hf_all, hf_tex, hf_rows)
assert len(XL_lf_tr) == N_LF_TRAIN and len(XL_hf_tr) == N_HF_TRAIN

# all distances are measured to the union of LF and HF training textures
tr_index = np.union1d(lf_index_vals[lf_tex], hf_index_vals[hf_tex])
X_tr_obs = X_lf_all[np.isin(lf_index_vals, tr_index)]
assert len(X_tr_obs) == len(tr_index), "every HF index must exist in the LF pool"
mu_d = X_tr_obs.mean(0)
sd_d = X_tr_obs.std(0)
sd_d[sd_d < 1e-12] = 1.0
Z_tr_d = (X_tr_obs - mu_d) / sd_d

hf_te = np.setdiff1d(np.arange(len(X_hf_all)), hf_tex)
N_HF_TEST = len(hf_te)
assert N_HF_TEST >= 2 * N_HF_SUB, "not enough HF test textures for distinct rand/far subsets"

assert len(np.intersect1d(hf_tex, hf_te)) == 0, "HF train overlaps HF test"

X_hf_te, Y_hf_te = X_hf_all[hf_te], Y_hf_all[hf_te]
hf_te_index = hf_index_vals[hf_te]
hf_te_dist = nearest_dist((X_hf_te - mu_d) / sd_d, Z_tr_d)
hf_te_dist_hf = nearest_dist((X_hf_te - mu_d) / sd_d, (X_hf_all[hf_tex] - mu_d) / sd_d)

hf_mask_far = np.zeros(N_HF_TEST, bool)
hf_mask_far[np.argsort(hf_te_dist_hf if FAR_REFERENCE == "hf" else hf_te_dist)[-N_HF_SUB:]] = True
hf_mask_rand = np.zeros(N_HF_TEST, bool)
hf_mask_rand[rng_hf_sub.choice(N_HF_TEST, N_HF_SUB, replace=False)] = True

hf_masks = {"all": np.ones(N_HF_TEST, bool), f"rand{N_HF_SUB}": hf_mask_rand,
            f"far{N_HF_SUB}": hf_mask_far}

def to_long(X):
    return np.column_stack([np.repeat(X, N_RHO, axis=0), np.tile(RHO_ARR, len(X))])

def long_to_wide(y_by_target, n):
    wide = np.empty((n, len(targ_cols)))
    wide[:, 0::2] = y_by_target["minor"].reshape(n, N_RHO)
    wide[:, 1::2] = y_by_target["major"].reshape(n, N_RHO)
    return wide

XL_hf_te = to_long(X_hf_te)

x_mu = XL_lf_tr.mean(0)
x_sd = XL_lf_tr.std(0)
x_sd[x_sd < 1e-12] = 1.0

def scale_x(X):
    return (X - x_mu) / x_sd

XsL_lf_tr, XsL_hf_tr = scale_x(XL_lf_tr), scale_x(XL_hf_tr)
XsL_hf_te = scale_x(XL_hf_te)

# minor strain at rho = 0 is identically zero, that column is never scored
ACTIVE = Y_lf_all.std(0) > 1e-12

RUN_INFO = {"tag": TAG, "repr": REPR, "strategy": STRATEGY, "repeat": REPEAT,
          "formulation": FORMULATION, "kernel_set": KERNEL_SET,
          "test_seed": TEST_SEED, "train_seed": TRAIN_SEED, "seed": SEED,
          "n_lf_train_rows": N_LF_TRAIN, "n_hf_train_rows": N_HF_TRAIN,
          "n_lf_train_textures": int(len(lf_tex)), "n_hf_train_textures": int(len(hf_tex)),
          "n_train_textures_union": int(len(tr_index)),
          "distance_reference": "LF_train U HF_train textures",
          "far_reference": FAR_REFERENCE,
          "lf_train_pairs": [[int(lf_index_vals[lf_tex][p]), float(RHO_ARR[k])] for p, k in lf_rows],
          "hf_train_pairs": [[int(hf_index_vals[hf_tex][p]), float(RHO_ARR[k])] for p, k in hf_rows],
          "n_hf_test": int(N_HF_TEST), "n_hf_sub": N_HF_SUB, "hf_train_texture_cap": int(HF_CAP),
          "hf_test_index": hf_te_index.tolist()}

print(f"LF train {N_LF_TRAIN} rows / {len(lf_tex)} textures, HF train {N_HF_TRAIN} rows / {len(hf_tex)} textures, "
      f"HF test {N_HF_TEST} textures")

In [ ]:
def score(model, level, y_true, y_mean, masks):
    out = []
    for name, m in masks.items():
        rmse = np.sqrt(((y_mean[m] - y_true[m]) ** 2).mean(0))
        out.append({"model": model, "level": level, "subset": name, "n": int(m.sum()),
                    "rmse_mean": float(rmse[ACTIVE].mean())})
    return out

def write_predictions(path, index, dist, mean, std):
    df = pd.DataFrame({"index": index, "nn_dist": dist})
    for j, c in enumerate(targ_cols):
        df[c] = mean[:, j]
    for j, c in enumerate(targ_cols):
        df[c + "_std"] = std[:, j]
    df.to_csv(path, index=False)

In [ ]:
import torch
from tabpfn import TabPFNRegressor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = ROOT / "tabpfn_3" / "tabpfn-v3-regressor-v3_default.ckpt"
QUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
print("device:", DEVICE, "| checkpoint:", CKPT.name)

def new_regressor():
    return TabPFNRegressor(device=DEVICE, model_path=str(CKPT), random_state=SEED)

def bar_variance(out):
    crit, logits = out["criterion"], out["logits"]
    try:
        var = crit.variance(logits)
    except RuntimeError:
        # logits and bar-distribution borders can come in different dtypes
        var = crit.variance(logits.to(crit.borders.dtype))
    return var.detach().cpu().numpy().astype(float)

def predict_dist(model, X):
    out = model.predict(X, output_type="full")
    mean = np.asarray(out["mean"], float)
    q = out["quantiles"]
    if len(q) != len(QUANTILES):
        q = model.predict(X, output_type="quantiles", quantiles=QUANTILES)
    return mean, bar_variance(out), np.column_stack([np.asarray(qi, float) for qi in q])

def fire_one_target(t):
    # LF model on the LF rows. Its predictive distribution at the HF inputs becomes extra
    # features for a second model that learns the HF residual. Raw strain targets throughout.
    lf = new_regressor().fit(XsL_lf_tr, yL_lf_tr[t])
    mu_tr, var_tr, q_tr = predict_dist(lf, XsL_hf_tr)
    Z_tr = np.hstack([XsL_hf_tr, mu_tr[:, None], var_tr[:, None], q_tr])
    dm = new_regressor().fit(Z_tr, yL_hf_tr[t] - mu_tr)
    mu_te, var_te, q_te = predict_dist(lf, XsL_hf_te)
    Z_te = np.hstack([XsL_hf_te, mu_te[:, None], var_te[:, None], q_te])
    r_out = dm.predict(Z_te, output_type="full")
    hf_mean = mu_te + np.asarray(r_out["mean"], float)
    hf_var = var_te + bar_variance(r_out)
    return hf_mean, np.sqrt(np.maximum(hf_var, 0))

pred = {t: fire_one_target(t) for t in TARGETS}
print("FIRE done")

fire_hf_mean = long_to_wide({t: pred[t][0] for t in TARGETS}, N_HF_TEST)
fire_hf_std = long_to_wide({t: pred[t][1] for t in TARGETS}, N_HF_TEST)

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# if the GP notebook already ran this config, check that both used the same split
gp_json = OUT_DIR / "ar1_nargp_results.json"
if gp_json.exists():
    gp_cfg = json.load(open(gp_json))["config"]
    for k in ("lf_train_pairs", "hf_train_pairs", "hf_test_index", "far_reference"):
        assert gp_cfg[k] == RUN_INFO[k], f"{k}: split differs from the GP notebook"
    print("same split as", gp_json.name)

summaries = score("FIRE", "HF_test", Y_hf_te, fire_hf_mean, hf_masks)
print(pd.DataFrame(summaries).to_string(index=False))

write_predictions(OUT_DIR / "pred_fire_hf_test.csv", hf_te_index, hf_te_dist, fire_hf_mean, fire_hf_std)

with open(OUT_DIR / "fire_results.json", "w") as f:
    json.dump({"config": RUN_INFO, "summaries": summaries}, f, indent=2)
print("written:", OUT_DIR)

In [ ]:
n_show = 4
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4), sharey=True)
for k, ax in enumerate(axes):
    for y_row, label, col in ((Y_hf_te[k], "HF true", "k"), (fire_hf_mean[k], "FIRE", "tab:green")):
        ax.plot(y_row[0::2], y_row[1::2], marker="o", ms=4, label=label, color=col)
    ax.set_title(f"HF test index={hf_te_index[k]}")
    ax.set_xlabel("minor strain")
axes[0].set_ylabel("major strain"); axes[0].legend()
fig.suptitle(f"{TAG}: HF test FLD curves")
fig.tight_layout()
plt.show()